In [10]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

c:\Dev\Projects\EuropeanEnergyDataAnalytics


In [11]:
import json
import boto3
import pandas as pd

from config.settings import load_config

config = load_config()

session = boto3.Session(
    profile_name=config["aws"]["profile_name"],
    region_name=config["project"]["region"],
)

s3 = session.client("s3")

bucket = config["project"]["bucket"]
bronze_prefix = config["storage"]["bronze_prefix"]

In [12]:
def load_bronze_dataset(dataset_name: str) -> pd.DataFrame:
    prefix = f"{bronze_prefix}/{dataset_name}/"

    response = s3.list_objects_v2(
        Bucket=bucket,
        Prefix=prefix,
    )

    objects = response.get("Contents", [])

    if not objects:
        raise FileNotFoundError(
            f"No Bronze file found for {dataset_name}"
        )

    latest_object = max(
        objects,
        key=lambda obj: obj["LastModified"],
    )

    key = latest_object["Key"]

    print(f"Loading: s3://{bucket}/{key}")

    response = s3.get_object(
        Bucket=bucket,
        Key=key,
    )

    raw = json.loads(
        response["Body"].read().decode("utf-8")
    )

    return pd.DataFrame(raw["data"])

In [13]:
datasets = {}

for dataset_name in config["datasets"]:
    datasets[dataset_name] = load_bronze_dataset(
        dataset_name
    )

Loading: s3://european-energy-data-analytics-488658242500-eu-central-1-an/bronze/ember/generation/ingestion_date=2026-09-11/generation_20260911T101523Z.json
Loading: s3://european-energy-data-analytics-488658242500-eu-central-1-an/bronze/ember/demand/ingestion_date=2026-09-11/demand_20260911T101537Z.json
Loading: s3://european-energy-data-analytics-488658242500-eu-central-1-an/bronze/ember/emissions/ingestion_date=2026-09-11/emissions_20260911T101543Z.json
Loading: s3://european-energy-data-analytics-488658242500-eu-central-1-an/bronze/ember/carbon_intensity/ingestion_date=2026-09-11/carbon_intensity_20260911T101557Z.json
Loading: s3://european-energy-data-analytics-488658242500-eu-central-1-an/bronze/ember/capacity/ingestion_date=2026-09-11/capacity_20260911T101558Z.json


In [14]:
for name, df in datasets.items():
    print(
        f"{name:20} "
        f"rows={len(df):>8,} "
        f"columns={len(df.columns):>2}"
    )

generation           rows=  43,900 columns= 8
demand               rows=   2,820 columns= 5
emissions            rows=  43,900 columns= 8
carbon_intensity     rows=   2,820 columns= 5
capacity             rows=   5,268 columns= 8


In [15]:
profile_rows = []

for dataset_name, df in datasets.items():
    for column in df.columns:
        profile_rows.append({
            "dataset": dataset_name,
            "column": column,
            "dtype": str(df[column].dtype),
            "row_count": len(df),
            "null_count": int(df[column].isna().sum()),
            "null_pct": round(df[column].isna().mean() * 100, 2),
            "unique_values": int(df[column].nunique(dropna=True)),
            "sample_values": ", ".join(
                map(
                    str,
                    df[column]
                    .dropna()
                    .astype(str)
                    .unique()[:3]
                )
            )
        })

profile_df = pd.DataFrame(profile_rows)

profile_df

,dataset,column,dtype,row_count,null_count,null_pct,unique_values,sample_values
0,generation,entity,str,43900,0,0.0,20,"Ireland, Austria, Belgium"
1,generation,entity_code,str,43900,0,0.0,20,"IRL, AUT, BEL"
2,generation,is_aggregate_entity,bool,43900,0,0.0,1,False
3,generation,date,str,43900,0,0.0,200,"2010-01-01, 2010-02-01, 2010-03-01"
4,generation,series,str,43900,0,0.0,17,"Bioenergy, Clean, Coal"
5,generation,is_aggregate_series,bool,43900,0,0.0,2,"False, True"
6,generation,generation_twh,float64,43900,0,0.0,3485,"0.0, 0.34, 0.4"
7,generation,share_of_generation_pct,float64,43900,0,0.0,10353,"0.0, 13.74, 15.93"
8,demand,entity,str,2820,0,0.0,20,"Ireland, Austria, Belgium"
9,demand,entity_code,str,2820,0,0.0,20,"IRL, AUT, BEL"


In [16]:
output_path = (
    PROJECT_ROOT
    / "docs"
    / "data_dictionary"
    / "bronze_data_profile.csv"
)

profile_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8",
)

print(f"Saved to: {output_path}")

Saved to: c:\Dev\Projects\EuropeanEnergyDataAnalytics\docs\data_dictionary\bronze_data_profile.csv


In [17]:
key_checks = {
    "generation": ["entity_code", "date", "series"],
    "demand": ["entity_code", "date"],
    "emissions": ["entity_code", "date", "series"],
    "carbon_intensity": ["entity_code", "date"],
    "capacity": ["entity_code", "date", "series"],
}

for name, key_columns in key_checks.items():
    df = datasets[name]

    duplicate_count = df.duplicated(
        subset=key_columns
    ).sum()

    print(
        f"{name:20} "
        f"duplicates={duplicate_count:>5} | "
        f"countries={df['entity_code'].nunique():>2} | "
        f"min={df['date'].min()} | "
        f"max={df['date'].max()}"
    )

generation           duplicates=    0 | countries=20 | min=2010-01-01 | max=2026-08-01
demand               duplicates=    0 | countries=20 | min=2010-01-01 | max=2026-08-01
emissions            duplicates=    0 | countries=20 | min=2010-01-01 | max=2026-08-01
carbon_intensity     duplicates=    0 | countries=20 | min=2010-01-01 | max=2026-08-01
capacity             duplicates=    0 | countries=12 | min=2016-01-01 | max=2026-08-01


In [18]:
capacity = datasets["capacity"]

print(
    f"Rows      : {len(capacity):,}\n"
    f"Countries : {capacity['entity_code'].nunique()}\n"
    f"Series    : {capacity['series'].nunique()}\n"
    f"Min date  : {capacity['date'].min()}\n"
    f"Max date  : {capacity['date'].max()}\n"
)

print("Countries:")
print(sorted(capacity["entity_code"].unique()))

print("\nSeries:")
print(sorted(capacity["series"].unique()))

Rows      : 5,268
Countries : 12
Series    : 5
Min date  : 2016-01-01
Max date  : 2026-08-01

Countries:
['BEL', 'DEU', 'DNK', 'ESP', 'FIN', 'FRA', 'GBR', 'HUN', 'ITA', 'NLD', 'POL', 'PRT']

Series:
['Offshore wind', 'Onshore wind', 'Solar', 'Unknown wind', 'Wind']


In [19]:
from src.quality.data_quality import validate_dataset

In [20]:
quality_results = []

for dataset_name, df in datasets.items():
    result = validate_dataset(
        df=df,
        dataset_name=dataset_name,
        expected_countries=config["countries"],
    )

    quality_results.append(result)

    print(
        f"{dataset_name:20} "
        f"passed={result['passed']} | "
        f"duplicates={result.get('duplicate_keys')} | "
        f"unexpected_countries={result.get('unexpected_countries')}"
    )

generation           passed=True | duplicates=0 | unexpected_countries=[]
demand               passed=True | duplicates=0 | unexpected_countries=[]
emissions            passed=True | duplicates=0 | unexpected_countries=[]
carbon_intensity     passed=True | duplicates=0 | unexpected_countries=[]
capacity             passed=True | duplicates=0 | unexpected_countries=[]
